# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [3]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json

import sys
sys.path.insert(0,'c:\\Users\\teede\\projects\\llm_engineering\\week1')

from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [4]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [5]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [1]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [ ]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'capabilities page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'capabilities page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'social profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 7 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'background / qualifications',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter/X profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [13]:
select_relevant_links("https://schermclubdenbosch.nl")

Selecting relevant links for https://schermclubdenbosch.nl by calling gpt-5-nano
Found 17 relevant links


{'links': [{'type': 'home page', 'url': 'https://www.schermclubdenbosch.nl'},
  {'type': 'about page',
   'url': 'https://www.schermclubdenbosch.nl/over-de-club/'},
  {'type': 'competitions page',
   'url': 'https://www.schermclubdenbosch.nl/over-de-club/wedstrijden/'},
  {'type': 'club clothing page',
   'url': 'https://www.schermclubdenbosch.nl/over-de-club/clubkleding/'},
  {'type': 'fencing equipment page',
   'url': 'https://www.schermclubdenbosch.nl/over-schermen/schermuitrusting/'},
  {'type': 'membership page',
   'url': 'https://www.schermclubdenbosch.nl/lid-worden/'},
  {'type': 'news hub', 'url': 'https://www.schermclubdenbosch.nl/nieuws/'},
  {'type': 'gallery hub',
   'url': 'https://www.schermclubdenbosch.nl/nieuws-fotos/'},
  {'type': 'news articles page',
   'url': 'https://www.schermclubdenbosch.nl/nieuws-fotos/nieuws/'},
  {'type': 'gallery photos page',
   'url': 'https://www.schermclubdenbosch.nl/nieuws-fotos/fotos/'},
  {'type': 'contact page',
   'url': 'https://w

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [15]:
print(fetch_page_and_all_relevant_links("https://schermclubdenbosch.nl"))

Selecting relevant links for https://schermclubdenbosch.nl by calling gpt-5-nano
Found 16 relevant links
## Landing Page:

Schermclub den Bosch voor topsporters en recreatieve sporters

Over de club
Wedstrijden
Clubkleding
Over schermen
Schermuitrusting
Lid worden
Nieuws & Foto’s
Nieuws
Foto’s
Contact
Over de club
Wedstrijden
Clubkleding
Over schermen
Schermuitrusting
Lid worden
Nieuws & Foto’s
Nieuws
Foto’s
Contact
WELKOM BIJ SCHERMCLUB DEN BOSCH
Schermclub Den Bosch is een degen schermclub voor topsporters en recreatieve sporters. We bieden groepstrainingen en individuele trainingen aan voor meer dan 80 leden. De trainingen vinden plaats in sportcomplex De Plek (voorheen Flik-Flak).
Jong en oud traint bij ons en iedereen krijgt voldoende aandacht. Maître Roel Verwijlen (voormalig bondcoach) en (jeugd)trainers Cheryl de Jong en Konrad Veenenbos verzorgen de trainingen. Kinderen vanaf 7 jaar zijn welkom om kennis  te maken met de schermsport.
Schermclub Den Bosch is lid van de Koninkli

In [17]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [18]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [20]:
get_brochure_user_prompt("Organon", "https://organon.com")

Selecting relevant links for https://organon.com by calling gpt-5-nano
Found 31 relevant links


"\nYou are looking at a company called: Organon\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHome | Organon\n\nSkip to content\nClose\nMenu\nClose\nMain menu\nAbout us\nOur products\nOur focus\nProducts list\nPatient support programs\nOrganon Pro for healthcare professionals\nSafety data sheets\nProduct patents\nOur leaders\nLeadership team\nBoard of Directors\nCorporate governance\nBusiness development\nOur culture\nOur stories\nPolicies & positions\nESG\nEnvironmental, social & governance\nSuppliers\nOur Suppliers\nSupplier relationship management\nMedia\nMedia\nNews\nInvestor relations\nInvestor relations overview\nEvents & presentations\nSEC filings\nFinancial Information\nInvestor resources\nCareers\nContact us\nGlobal\nOpens a new window\nOpens a new window\nOpens a new window\nOpens a new window\nOpens a new window\nMain menu\nOrganon s

In [21]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
create_brochure("Organon", "https://organon.com")

Selecting relevant links for https://organon.com by calling gpt-5-nano
Found 27 relevant links


# Welcome to Organon: Here for Her Health — and Maybe a Laugh or Two!

---

## Who We Are

Organon is not your average healthcare company — we’re a global, independent champion focused **solely on women's health**. Because let's face it, women deserve a better and healthier every day, every day, everywhere.

Founded with a bold vision to improve women's lives throughout every stage, we proudly offer a **diverse portfolio of over 70 medicines and devices**. Our specialties? Women's health (obviously), biosimilars, and well-established trusted brands that cover a wide spectrum of conditions and diseases.

---

## What We’re About (Besides Saving the World)

- **Her health, her future:** From the first period jitters to menopause mysteries, we're here with treatments and support programs designed specifically for women’s complex health needs.
- **Global reach:** With operations spanning The Americas, Europe, Asia Pacific, and Middle East & Africa, we’re practically everywhere you want us to be.
- **Going green:** We don’t just care for women, we care for the planet, embracing ESG (Environmental, Social & Governance) principles like pros.

---

## Our Culture: Serious About Care, Lighthearted About Everything Else

People often ask what it’s like working at Organon. We say, picture a place where:

- You can **make a difference every single day**, because your work literally improves lives.
- Innovation and empathy go hand-in-hand — no stiff white coats here, just passionate folks united for a cause.
- Diversity, inclusion, and respect aren’t buzzwords — they’re how we roll.
- There’s room to grow, lead, and yes, even crack a joke or two during meetings.
  
If you think healthcare is all sterile lab coats and buzzwords, think again. At Organon, our culture sparkles as much as our science.

---

## Calling All Future Talent: Careers at Organon

Looking for a career that blends meaningful impact with a fun and inclusive vibe? Organon is hiring globally across multiple functions, from research and development, business development, to marketing and supply chain.

Perks include:

- **Purpose-driven work:** Because saving and improving lives beats the daily grind.
- **Diverse and global teams:** Show off your language skills — we operate in over 40 countries with multiple languages.
- **Growth opportunities:** Whether you want to innovate scientific breakthroughs or lead global strategies, it's all within reach.

---

## For Our Customers and Healthcare Professionals

Organon offers more than just products; we provide comprehensive patient support programs and professional resources through **Organon Pro** to ensure therapies are delivered with care and knowledge.

With patents and safety data sheets readily available, alongside a commitment to transparency and quality, we strive to build trusted partnerships worldwide.

---

## Investors, Here’s the Skinny

We’re on a steady growth trajectory, riding the wave of innovation in women’s healthcare and biosimilars with a robust portfolio. Our governance and business development teams keep us nimble and focused, ensuring that every move drives long-term value.

---

## Final Thought: Why Choose Organon?

Because in a world full of healthcare companies, Organon is the one saying, “We got her back... and her front, too.” We’re for women, by women, and with science, heart, and humor.

Ready to join the movement? Whether you’re a patient, professional, investor, or future team member, Organon welcomes you to a healthier tomorrow.

---

*Organon — improving women’s health, one laugh and one life at a time.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>